# Scheduling End-to-End Debug

This notebook is a standalone debug harness for the optimization pipeline.

It does not import from `scheduling/`. Instead it rebuilds the critical pieces locally so you can see where the formulation first breaks:

1. feature reconstruction from the data-generation extraction blocks,
2. feature-contract comparison for `tmp` vs root,
3. scaled input and output constraints,
4. neural-network embedding,
5. base-case line constraints,
6. preventive N-1 constraints,
7. staged solve ladder to isolate the first failing block.


In [ ]:
from pathlib import Path
import csv
import sys

import andes
import cvxpy as cp
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml

CURRENT = Path.cwd().resolve()
REPO_ROOT = CURRENT
while REPO_ROOT.name != "vis-ml" and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if REPO_ROOT.name != "vis-ml":
    raise RuntimeError("Could not locate the vis-ml repository root.")

DATA_GEN_DIR = REPO_ROOT / "data_generation"
if str(DATA_GEN_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_GEN_DIR))

from extract_metrics import (
    extract_line_metrics,
    extract_operating_point_snapshot,
    extract_x_cont,
    extract_x_op,
    extract_x_sched,
)
from models.models import create_model
from models.convex_models import NonNegLinear

from andes.interop.pandapower import to_pandapower
from pandapower import auxiliary as aux
from pandapower.pd2ppc import _pd2ppc
from pandapower.pypower.makePTDF import makePTDF
from pandapower.pypower.makeLODF import makeLODF

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


In [ ]:
CONFIG = {
    "system_case": REPO_ROOT / "data_generation/andes_cases/ieee39_full_ibrs.xlsx",
    "feature_names_path": REPO_ROOT / "configs/data_generation_feature_names.yaml",
    "tmp_cfg": REPO_ROOT / "tmp/optimization/optimization.yaml",
    "tmp_sched_cfg": REPO_ROOT / "tmp/scheduling/mtlsh_convex.yaml",
    "root_cfg": REPO_ROOT / "results/thesis_optimization_results/configs/base_optimization_mtlsh.yaml",
    "root_model_run_config": REPO_ROOT / "results/thesis_model_results/exports/optimization_ready_mtlsh/artifacts/run_config.yaml",
    "base_scale": 0.6,
    "step_scale": 1.2,
    "load_step_time": 3.0,
    "seed_M": 4.0,
    "seed_D": 2.0,
    "solver_continuous": "OSQP",
    "solver_mip": "GUROBI",
}

def load_yaml(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

tmp_cfg = load_yaml(CONFIG["tmp_cfg"])
tmp_sched_cfg = load_yaml(CONFIG["tmp_sched_cfg"])
root_cfg = load_yaml(CONFIG["root_cfg"])
root_model_run_cfg = load_yaml(CONFIG["root_model_run_config"])

TMP_X_FEATURES = list((tmp_cfg.get("features") or {}).get("x_features") or [])
TMP_Y_NAMES = list((tmp_sched_cfg.get("features") or {}).get("y_features") or root_cfg.get("outputs", {}).get("y_names", []))
ROOT_X_FEATURES = list(((root_model_run_cfg.get("resolved") or {}).get("feature_cols") or []))
ROOT_Y_NAMES = list((root_cfg.get("outputs") or {}).get("y_names") or [])

print("tmp x feature count:", len(TMP_X_FEATURES))
print("root x feature count:", len(ROOT_X_FEATURES))
print("tmp y names:", TMP_Y_NAMES)
print("root y names:", ROOT_Y_NAMES)


In [ ]:
def sanitize_owner_label(value) -> str:
    text = str(value).strip()
    text = "_".join(part for part in text.replace("/", " ").replace("-", " ").split())
    return text or "owner"

def build_post_step_p_vector(ss, *, step_scale: float, target_pq_names=None, target_owner_labels=None):
    target_names = set(str(v) for v in (target_pq_names or []))
    target_owners = set(str(v) for v in (target_owner_labels or []))
    actual_names = [str(v) for v in list(ss.PQ.name.v)]
    actual_owners = [sanitize_owner_label(v) for v in list(ss.PQ.owner.v)]
    p_before = np.asarray(ss.PQ.p0.v, dtype=float).copy()
    p_after = p_before.copy()
    explicit_targeting = bool(target_names) or bool(target_owners)
    if not explicit_targeting:
        targets = set(actual_names)
    else:
        targets = set()
        for name, owner in zip(actual_names, actual_owners):
            if name in target_names or owner in target_owners:
                targets.add(name)
    for idx, name in enumerate(actual_names):
        if name in targets:
            p_after[idx] = float(p_after[idx]) * float(step_scale)
    return p_after

def derive_sched_dispatch_vectors(ss):
    dispatch = np.asarray(list(ss.PV.p0.v) + list(ss.Slack.p0.v), dtype=float).reshape(-1)
    n_dispatch = int(dispatch.size)
    ibr_positions = []
    gen_values = getattr(getattr(ss.REGCV1, "gen", None), "v", None)
    if gen_values is not None:
        for value in list(gen_values):
            try:
                pos = int(value) - 1
            except Exception:
                continue
            if 0 <= pos < n_dispatch and pos not in ibr_positions:
                ibr_positions.append(pos)
    if not ibr_positions:
        ibr_positions = list(range(min(int(ss.REGCV1.n), n_dispatch)))

    regcv1_pg = np.full(int(ss.REGCV1.n), np.nan, dtype=float)
    for local_idx, pos in enumerate(ibr_positions[: int(ss.REGCV1.n)]):
        regcv1_pg[local_idx] = float(dispatch[pos])

    genrou_positions = [idx for idx in range(n_dispatch) if idx not in set(ibr_positions)]
    genrou_pg = np.full(int(getattr(ss.GENROU, "n", 0)), np.nan, dtype=float)
    for local_idx, pos in enumerate(genrou_positions[: int(getattr(ss.GENROU, "n", 0))]):
        genrou_pg[local_idx] = float(dispatch[pos])
    return genrou_pg, regcv1_pg

def build_features_from_datagen(
    ss,
    *,
    base_scale: float,
    step_scale: float,
    load_step_time: float,
    M_vec,
    D_vec,
    contingency=None,
    feature_names_path=None,
    load_step_target_pq_names=None,
    load_step_target_owners=None,
):
    pq_names = list(ss.PQ.name.v) if getattr(ss, "PQ", None) is not None and ss.PQ.n > 0 else []
    pq_owners = [sanitize_owner_label(v) for v in list(ss.PQ.owner.v)] if pq_names else []
    pq_p_before = np.asarray(ss.PQ.p0.v, dtype=float).copy() if pq_names else np.zeros(0, dtype=float)
    pq_q_before = np.asarray(ss.PQ.q0.v, dtype=float).copy() if pq_names else np.zeros(0, dtype=float)
    pq_p_after = build_post_step_p_vector(
        ss,
        step_scale=step_scale,
        target_pq_names=load_step_target_pq_names,
        target_owner_labels=load_step_target_owners,
    ) if pq_names else np.zeros(0, dtype=float)
    line_uids = list(range(int(getattr(getattr(ss, "Line", None), "n", 0))))

    operating_snapshot = extract_operating_point_snapshot(ss)
    line_metrics_snapshot = extract_line_metrics(
        ss=ss,
        contingency=dict(contingency) if contingency is not None else None,
        line_uids=line_uids,
        feature_names_path=str(feature_names_path) if feature_names_path is not None else None,
    )

    x_op = extract_x_op(
        ss=ss,
        base_load_scale=float(base_scale),
        pq_names=pq_names,
        pq_owners=pq_owners,
        pq_p_before=pq_p_before,
        pq_q_before=pq_q_before,
        operating_point_snapshot=operating_snapshot,
        feature_names_path=str(feature_names_path) if feature_names_path is not None else None,
    )

    x_cont = extract_x_cont(
        ss=ss,
        contingency=dict(contingency) if contingency is not None else None,
        load_step_scale=float(step_scale),
        load_step_time=float(load_step_time),
        pq_names=pq_names,
        pq_owners=pq_owners,
        pq_p_before=pq_p_before,
        pq_p_after=pq_p_after,
        line_uids=line_uids,
        line_metrics_snapshot=line_metrics_snapshot,
        feature_names_path=str(feature_names_path) if feature_names_path is not None else None,
    )

    genrou_pg, regcv1_pg = derive_sched_dispatch_vectors(ss)
    x_sched = extract_x_sched(
        ss=ss,
        M_vec=M_vec,
        D_vec=D_vec,
        genrou_pg=genrou_pg,
        regcv1_pg=regcv1_pg,
        feature_names_path=str(feature_names_path) if feature_names_path is not None else None,
    )

    row = {}
    row.update(x_op)
    for section in ("load_mismatch", "line_identity", "line_flow", "line_bus", "line_severity"):
        row.update(dict(x_cont.get(section, {}) or {}))
    row.update(x_sched)
    return row


In [ ]:
def resolve_model_dir(model_cfg):
    raw = str(model_cfg.get("model_dir", model_cfg.get("state_dict", ""))).strip()
    if not raw:
        raise ValueError("Missing model path in config.")
    path = Path(raw)
    if not path.is_absolute():
        path = (REPO_ROOT / path).resolve()
    return path.parent if path.suffix else path

def resolve_state_dict_path(model_dir: Path):
    candidates = [
        model_dir / "vis_mlp_state_dict_best.pt",
        model_dir / "mtlsh_state_dict_best.pt",
        model_dir / "mlp_state_dict_best.pt",
        model_dir / "state_dict_best.pt",
        model_dir / "state_dict.pt",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No state dict found in {model_dir}")

def load_model_and_scalers(model_cfg, *, run_cfg=None):
    model_dir = resolve_model_dir(model_cfg)
    merged_cfg = dict(model_cfg)
    if run_cfg is not None:
        resolved = dict((run_cfg.get("resolved") or {}))
        data = dict((run_cfg.get("data") or {}))
        if not merged_cfg.get("in_dim") and resolved.get("feature_cols"):
            merged_cfg["in_dim"] = len(resolved["feature_cols"])
        if not merged_cfg.get("n_tasks") and data.get("target_cols"):
            merged_cfg["n_tasks"] = len(data["target_cols"])
    model, _ = create_model(
        model_type=str(merged_cfg.get("type", "")).replace("MTLSharedHeads", "MTLSH"),
        in_dim=int(merged_cfg["in_dim"]),
        out_dim=int(merged_cfg.get("out_dim", merged_cfg.get("n_tasks", 0))),
        device="cpu",
        hidden_sizes=merged_cfg.get("hidden_sizes"),
        shared_sizes=merged_cfg.get("shared_sizes"),
        head_sizes=merged_cfg.get("head_sizes"),
        dropout=float(merged_cfg.get("dropout", 0.0)),
        u_feature_idx=merged_cfg.get("u_feature_idx"),
        v_feature_idx=merged_cfg.get("v_feature_idx"),
        activation=str(merged_cfg.get("activation", "relu")),
    )
    state_dict = torch.load(resolve_state_dict_path(model_dir), map_location="cpu")
    model.load_state_dict(state_dict)
    model.eval()
    x_scaler = joblib.load(model_dir / "x_scaler.pkl")
    y_scaler = joblib.load(model_dir / "y_scaler.pkl")
    return model, x_scaler, y_scaler, model_dir

def scale_raw_with_minmax(values, scaler):
    arr = np.asarray(values, dtype=float)
    return arr * scaler.scale_ + scaler.min_

def nn_linear_layers(module):
    return [m for m in module if isinstance(m, torch.nn.Linear)]

def extract_linear_layers(seq):
    layers = []
    for layer in nn_linear_layers(seq):
        weight = layer.weight
        if isinstance(layer, NonNegLinear):
            weight = F.softplus(weight)
        layers.append((weight.detach().cpu().numpy(), layer.bias.detach().cpu().numpy()))
    return layers

def interval_bounds(W, b, h_min, h_max):
    W_pos = np.maximum(W, 0)
    W_neg = np.minimum(W, 0)
    z_min = W_pos @ h_min + W_neg @ h_max + b
    z_max = W_pos @ h_max + W_neg @ h_min + b
    return z_min, z_max

def relu_milp(z, y, z_min, z_max, *, name):
    constraints = []
    active = z_min >= 0
    inactive = z_max <= 0
    uncertain = ~(active | inactive)
    if np.any(active):
        idx = np.flatnonzero(active)
        constraints += [y[idx] == z[idx], z[idx] >= 0]
    if np.any(inactive):
        idx = np.flatnonzero(inactive)
        constraints += [y[idx] == 0, z[idx] <= 0]
    if np.any(uncertain):
        idx = np.flatnonzero(uncertain)
        a = cp.Variable(idx.shape[0], boolean=True, name=f"{name}_bin")
        constraints += [
            y[idx] >= 0,
            y[idx] >= z[idx],
            y[idx] <= z[idx] - cp.multiply(z_min[idx], (1 - a)),
            y[idx] <= cp.multiply(z_max[idx], a),
        ]
    return constraints

def build_nn_constraints_local(model, x, x_min_sc, x_max_sc):
    if hasattr(model, "net"):
        constraints = []
        h = x
        h_min = x_min_sc.copy()
        h_max = x_max_sc.copy()
        layers = extract_linear_layers(model.net)
        for idx, (W, b) in enumerate(layers):
            z = W @ h + b
            if idx < len(layers) - 1:
                z_min, z_max = interval_bounds(W, b, h_min, h_max)
                y = cp.Variable(b.shape[0], name=f"mlp_{idx}")
                constraints += relu_milp(z, y, z_min, z_max, name=f"mlp_{idx}")
                h = y
                h_min = np.maximum(0, z_min)
                h_max = np.maximum(0, z_max)
            else:
                y_out = cp.Variable(b.shape[0], name="mlp_out")
                constraints.append(y_out == z)
        return y_out, constraints

    if hasattr(model, "shared") and hasattr(model, "heads"):
        constraints = []
        h = x
        h_min = x_min_sc.copy()
        h_max = x_max_sc.copy()
        for idx, (W, b) in enumerate(extract_linear_layers(model.shared)):
            z = W @ h + b
            z_min, z_max = interval_bounds(W, b, h_min, h_max)
            y = cp.Variable(b.shape[0], name=f"shared_{idx}")
            constraints += relu_milp(z, y, z_min, z_max, name=f"shared_{idx}")
            h = y
            h_min = np.maximum(0, z_min)
            h_max = np.maximum(0, z_max)

        outputs = []
        for head_idx, head in enumerate(model.heads):
            hh = h
            hh_min = h_min.copy()
            hh_max = h_max.copy()
            head_layers = extract_linear_layers(head)
            for layer_idx, (W, b) in enumerate(head_layers):
                z = W @ hh + b
                if layer_idx < len(head_layers) - 1:
                    z_min, z_max = interval_bounds(W, b, hh_min, hh_max)
                    y = cp.Variable(b.shape[0], name=f"head{head_idx}_{layer_idx}")
                    constraints += relu_milp(z, y, z_min, z_max, name=f"head{head_idx}_{layer_idx}")
                    hh = y
                    hh_min = np.maximum(0, z_min)
                    hh_max = np.maximum(0, z_max)
                else:
                    y_out = cp.Variable(1, name=f"out_{head_idx}")
                    constraints.append(y_out == z)
                    outputs.append(y_out)
        return cp.hstack(outputs), constraints

    raise NotImplementedError("Unsupported model type for local NN builder")


In [ ]:
def sanitize_branch_fmax_local(ss, branch, *, sentinel_threshold=1.0e4):
    fmax_raw = np.asarray(np.real(branch[:, 5]), dtype=float).copy()
    valid = np.isfinite(fmax_raw) & (fmax_raw > 0) & (fmax_raw < float(sentinel_threshold))
    if not np.all(valid):
        n_line = int(getattr(getattr(ss, "Line", None), "n", 0))
        rate_a_vals = list(getattr(getattr(getattr(ss, "Line", None), "rate_a", None), "v", []))
        sn_vals = list(getattr(getattr(getattr(ss, "Line", None), "Sn", None), "v", []))
        if branch.shape[0] == n_line and n_line > 0:
            for i in np.where(~valid)[0]:
                candidate = np.nan
                if i < len(rate_a_vals):
                    try:
                        candidate = float(rate_a_vals[i])
                    except Exception:
                        candidate = np.nan
                if not (np.isfinite(candidate) and candidate > 0 and candidate < sentinel_threshold):
                    if i < len(sn_vals):
                        try:
                            candidate = float(sn_vals[i])
                        except Exception:
                            candidate = np.nan
                if np.isfinite(candidate) and candidate > 0 and candidate < sentinel_threshold:
                    fmax_raw[i] = candidate
        valid = np.isfinite(fmax_raw) & (fmax_raw > 0) & (fmax_raw < float(sentinel_threshold))
        if not np.all(valid):
            raise RuntimeError("Invalid branch limits remain after sanitization.")
    return fmax_raw / ss.config.mva

def build_line_artifacts_local(ss):
    pp_net = to_pandapower(ss, verify=False)
    pp_net._options = {}
    aux._add_ppc_options(
        pp_net,
        calculate_voltage_angles=True,
        trafo_model="pi",
        check_connectivity=False,
        mode="opf",
        switch_rx_ratio=2,
        enforce_q_lims=False,
        recycle=None,
    )
    _, ppci = _pd2ppc(pp_net)
    branch = ppci["branch"]
    ptdf = np.asarray(makePTDF(ppci["baseMVA"], ppci["bus"], branch, using_sparse_solver=False), dtype=float)
    fmax = sanitize_branch_fmax_local(ss, branch)
    bus_ids = pp_net.bus.index.to_numpy(dtype=int)
    bus_df = ss.Bus.as_df()[["idx"]]
    bus_num_to_uid = {int(row.idx): int(uid) for uid, row in bus_df.iterrows()}
    bus_pos = {int(bus): i for i, bus in enumerate(bus_ids)}
    gen_buses = np.concatenate([np.asarray(ss.PV.bus.v, dtype=int), np.asarray(ss.Slack.bus.v, dtype=int)])
    load_buses = np.asarray(ss.PQ.bus.v, dtype=int)
    Cg = np.zeros((len(bus_ids), len(gen_buses)), dtype=float)
    Cd = np.zeros((len(bus_ids), len(load_buses)), dtype=float)
    for j, bus in enumerate(gen_buses):
        uid = bus_num_to_uid[int(bus)]
        Cg[bus_pos[uid], j] += 1.0
    for j, bus in enumerate(load_buses):
        uid = bus_num_to_uid[int(bus)]
        Cd[bus_pos[uid], j] += 1.0
    return {"ppci": ppci, "ptdf": ptdf, "fmax": fmax, "Cg": Cg, "Cd": Cd}

def build_basecase_line_constraints_local(pg, pd, artifacts):
    injections = artifacts["Cg"] @ pg - artifacts["Cd"] @ pd
    flows = artifacts["ptdf"] @ injections
    constraints = [flows <= artifacts["fmax"], flows >= -artifacts["fmax"]]
    return flows, constraints

def build_n1_constraints_local(flows, artifacts):
    branch = artifacts["ppci"]["branch"]
    try:
        lodf = np.asarray(makeLODF(branch, artifacts["ptdf"]), dtype=float)
    except TypeError:
        lodf = np.asarray(makeLODF(artifacts["ptdf"], branch), dtype=float)
    constraints = []
    active = 0
    for outage in range(len(artifacts["fmax"])):
        col = lodf[:, outage]
        if not np.all(np.isfinite(col)):
            continue
        f_post = flows + cp.multiply(col, flows[outage])
        constraints += [f_post <= artifacts["fmax"], f_post >= -artifacts["fmax"]]
        active += 1
    return constraints, active

def load_dispatch_cost_arrays_from_table(ss, cost_table_path: Path):
    with cost_table_path.open("r", encoding="utf-8") as f:
        payload = yaml.safe_load(f) or {}
    rows = list(payload.get("generators") or [])
    by_bus = {int(row["bus"]): row for row in rows}
    gen_buses = [int(v) for v in list(ss.PV.bus.v) + list(ss.Slack.bus.v)]
    a = np.zeros(len(gen_buses), dtype=float)
    b = np.zeros(len(gen_buses), dtype=float)
    c = np.zeros(len(gen_buses), dtype=float)
    for idx, bus in enumerate(gen_buses):
        row = by_bus[bus]
        a[idx] = float(row["a"])
        b[idx] = float(row["b"])
        c[idx] = float(row["c"])
    return a, b, c


In [ ]:
base_ss = andes.load(str(CONFIG["system_case"]), setup=False)
base_ss.config.freq = 50.0
for uid in range(base_ss.PQ.n):
    base_ss.PQ.p0.v[uid] = float(base_ss.PQ.p0.v[uid]) * float(CONFIG["base_scale"])
    base_ss.PQ.q0.v[uid] = float(base_ss.PQ.q0.v[uid]) * float(CONFIG["base_scale"])
for uid in range(base_ss.PV.n):
    base_ss.PV.p0.v[uid] = float(base_ss.PV.p0.v[uid]) * float(CONFIG["base_scale"])
    base_ss.PV.q0.v[uid] = float(base_ss.PV.q0.v[uid]) * float(CONFIG["base_scale"])
base_ss.PQ.config.p2p = 1
base_ss.PQ.config.q2q = 1
base_ss.PQ.config.p2z = 0
base_ss.PQ.config.q2z = 0
base_ss.PQ.config.p2i = 0
base_ss.PQ.config.q2i = 0
base_ss.PQ.config.pq2z = 0
base_ss.setup()

M_vec = np.full(int(base_ss.REGCV1.n), float(CONFIG["seed_M"]), dtype=float)
D_vec = np.full(int(base_ss.REGCV1.n), float(CONFIG["seed_D"]), dtype=float)
base_ss.REGCV1.M.v = M_vec.tolist()
base_ss.REGCV1.D.v = D_vec.tolist()

feature_row = build_features_from_datagen(
    base_ss,
    base_scale=CONFIG["base_scale"],
    step_scale=CONFIG["step_scale"],
    load_step_time=CONFIG["load_step_time"],
    M_vec=M_vec,
    D_vec=D_vec,
    feature_names_path=CONFIG["feature_names_path"],
)

comparison = pd.DataFrame({"feature": sorted(set(TMP_X_FEATURES) | set(ROOT_X_FEATURES))})
comparison["in_tmp"] = comparison["feature"].isin(TMP_X_FEATURES)
comparison["in_root"] = comparison["feature"].isin(ROOT_X_FEATURES)
comparison["buildable_from_datagen"] = comparison["feature"].isin(feature_row.keys())
comparison["root_missing_from_builder"] = comparison["in_root"] & ~comparison["buildable_from_datagen"]
display(pd.DataFrame([
    {"metric": "tmp x count", "value": int(comparison["in_tmp"].sum())},
    {"metric": "root x count", "value": int(comparison["in_root"].sum())},
    {"metric": "root missing from direct builder", "value": int(comparison["root_missing_from_builder"].sum())},
]))
display(comparison.loc[comparison["root_missing_from_builder"], ["feature"]].head(40))


In [ ]:
MODEL_BUNDLE = "root"  # choose: "tmp" or "root"

if MODEL_BUNDLE == "tmp":
    model_cfg = dict(tmp_cfg.get("model") or {})
    run_cfg = None
    x_features = TMP_X_FEATURES
    y_names = TMP_Y_NAMES
    y_min_raw = np.asarray((tmp_cfg.get("bounds") or {}).get("y_min") or [], dtype=float)
    y_max_raw = np.asarray((tmp_cfg.get("bounds") or {}).get("y_max") or [], dtype=float)
    ibr_idx = np.asarray((tmp_cfg.get("ibr") or {}).get("indices") or [], dtype=int)
    y_ibr_idx = np.asarray((tmp_cfg.get("constraints") or {}).get("y_ibr_idx") or [2, 3, 4, 5], dtype=int)
    cost_table_path = None
else:
    model_cfg = dict(root_cfg.get("model") or {})
    run_cfg = root_model_run_cfg
    x_features = ROOT_X_FEATURES
    y_names = ROOT_Y_NAMES
    y_min_raw = np.asarray((root_cfg.get("bounds") or {}).get("y_min") or [], dtype=float)
    y_max_raw = np.asarray((root_cfg.get("bounds") or {}).get("y_max") or [], dtype=float)
    ibr_idx = np.asarray((root_cfg.get("ibr") or {}).get("indices") or [], dtype=int)
    y_ibr_idx = np.asarray((root_cfg.get("constraints") or {}).get("y_ibr_idx") or [2, 3, 4, 5], dtype=int)
    raw_cost_table = str((root_cfg.get("ed_costs") or {}).get("cost_table_path", "")).strip()
    cost_table_path = (REPO_ROOT / raw_cost_table).resolve() if raw_cost_table else None

model, x_scaler, y_scaler, model_dir = load_model_and_scalers(model_cfg, run_cfg=run_cfg)
print("bundle:", MODEL_BUNDLE)
print("model dir:", model_dir)
print("n x features:", len(x_features))
print("y names:", y_names)


In [ ]:
def solve_debug_stage(
    *,
    use_input=True,
    use_output=False,
    use_nn=False,
    use_line=False,
    use_n1=False,
    balance_mode="tmp",
):
    ss = andes.load(str(CONFIG["system_case"]), setup=False)
    ss.config.freq = 50.0
    for uid in range(ss.PQ.n):
        ss.PQ.p0.v[uid] = float(ss.PQ.p0.v[uid]) * float(CONFIG["base_scale"])
        ss.PQ.q0.v[uid] = float(ss.PQ.q0.v[uid]) * float(CONFIG["base_scale"])
    for uid in range(ss.PV.n):
        ss.PV.p0.v[uid] = float(ss.PV.p0.v[uid]) * float(CONFIG["base_scale"])
        ss.PV.q0.v[uid] = float(ss.PV.q0.v[uid]) * float(CONFIG["base_scale"])
    ss.PQ.config.p2p = 1
    ss.PQ.config.q2q = 1
    ss.PQ.config.p2z = 0
    ss.PQ.config.q2z = 0
    ss.PQ.config.p2i = 0
    ss.PQ.config.q2i = 0
    ss.PQ.config.pq2z = 0
    ss.setup()

    M_seed = np.full(int(ss.REGCV1.n), float(CONFIG["seed_M"]), dtype=float)
    D_seed = np.full(int(ss.REGCV1.n), float(CONFIG["seed_D"]), dtype=float)
    ss.REGCV1.M.v = M_seed.tolist()
    ss.REGCV1.D.v = D_seed.tolist()

    feat = build_features_from_datagen(
        ss,
        base_scale=CONFIG["base_scale"],
        step_scale=CONFIG["step_scale"],
        load_step_time=CONFIG["load_step_time"],
        M_vec=M_seed,
        D_vec=D_seed,
        feature_names_path=CONFIG["feature_names_path"],
    )
    missing_features = [name for name in x_features if name not in feat]
    x_seed_raw = np.asarray([feat.get(name, np.nan) for name in x_features], dtype=float)
    nonfinite_seed = [x_features[i] for i in range(len(x_features)) if not np.isfinite(x_seed_raw[i])]

    pg_min = np.asarray(list(ss.PV.pmin.v) + list(ss.Slack.pmin.v), dtype=float)
    pg_max = np.asarray(list(ss.PV.pmax.v) + list(ss.Slack.pmax.v), dtype=float)
    pg_base = np.asarray(list(ss.PV.p0.v) + list(ss.Slack.p0.v), dtype=float)
    pd_tmp = np.asarray(ss.PQ.p0.v, dtype=float) * float(CONFIG["step_scale"])
    pd_root = build_post_step_p_vector(ss, step_scale=CONFIG["step_scale"])
    pd = pd_tmp if balance_mode == "tmp" else pd_root
    balance_rhs = float(np.sum(pd)) if balance_mode == "tmp" else float(np.sum(pd)) / float(CONFIG["step_scale"])

    m_idx = [i for i, name in enumerate(x_features) if name.startswith("M_") and name[2:].isdigit()]
    d_idx = [i for i, name in enumerate(x_features) if name.startswith("D_") and name[2:].isdigit()]
    pg_feature_idx = [i for i, name in enumerate(x_features) if name.startswith("P_GENROU_") or name.startswith("P_REGCV1_")]

    m_lo_raw = np.zeros(len(m_idx), dtype=float)
    m_hi_raw = np.full(len(m_idx), 8.0, dtype=float)
    d_lo_raw = np.zeros(len(d_idx), dtype=float)
    d_hi_raw = np.full(len(d_idx), 6.0, dtype=float)

    x_seed_sc = scale_raw_with_minmax(x_seed_raw, x_scaler)
    x_min_sc = x_seed_sc.copy()
    x_max_sc = x_seed_sc.copy()
    for k, idx in enumerate(m_idx):
        x_min_sc[idx] = m_lo_raw[k] * x_scaler.scale_[idx] + x_scaler.min_[idx]
        x_max_sc[idx] = m_hi_raw[k] * x_scaler.scale_[idx] + x_scaler.min_[idx]
    for k, idx in enumerate(d_idx):
        x_min_sc[idx] = d_lo_raw[k] * x_scaler.scale_[idx] + x_scaler.min_[idx]
        x_max_sc[idx] = d_hi_raw[k] * x_scaler.scale_[idx] + x_scaler.min_[idx]
    for k, idx in enumerate(pg_feature_idx[: min(len(pg_feature_idx), len(pg_min))]):
        x_min_sc[idx] = pg_min[k] * x_scaler.scale_[idx] + x_scaler.min_[idx]
        x_max_sc[idx] = pg_max[k] * x_scaler.scale_[idx] + x_scaler.min_[idx]

    y_min_sc = scale_raw_with_minmax(y_min_raw, y_scaler) if y_min_raw.size else np.zeros(0, dtype=float)
    y_max_sc = scale_raw_with_minmax(y_max_raw, y_scaler) if y_max_raw.size else np.zeros(0, dtype=float)

    pg = cp.Variable(len(pg_min), name="pg")
    x = cp.Variable(len(x_features), name="x") if (use_input or use_nn) else None
    y = cp.Variable(len(y_names), name="y") if (use_output or use_nn) else None

    constraints = [pg >= pg_min, pg <= pg_max, cp.sum(pg) == balance_rhs]
    block_sizes = {"ed": 3, "input": 0, "output": 0, "nn": 0, "line": 0, "n1": 0}

    if use_input or use_nn:
        if missing_features or nonfinite_seed:
            return {
                "status": "contract_error",
                "missing_features": missing_features,
                "nonfinite_seed": nonfinite_seed,
                "n_constraints": len(constraints),
                "block_sizes": block_sizes,
            }
        free = set(m_idx) | set(d_idx)
        if pg_feature_idx:
            free.update(pg_feature_idx[: min(len(pg_feature_idx), len(pg_min))])
        fixed_idx = np.asarray([i for i in range(len(x_features)) if i not in free], dtype=int)
        if fixed_idx.size:
            constraints.append(x[fixed_idx] == x_seed_sc[fixed_idx])
        if m_idx:
            constraints += [x[m_idx] >= x_min_sc[m_idx], x[m_idx] <= x_max_sc[m_idx]]
        if d_idx:
            constraints += [x[d_idx] >= x_min_sc[d_idx], x[d_idx] <= x_max_sc[d_idx]]
        if pg_feature_idx:
            link_idx = pg_feature_idx[: min(len(pg_feature_idx), len(pg_min))]
            constraints.append(x[link_idx] == cp.multiply(pg[: len(link_idx)], x_scaler.scale_[link_idx]) + x_scaler.min_[link_idx])
        block_sizes["input"] = len(constraints) - sum(v for k, v in block_sizes.items() if k != "input")

    if use_output:
        if y_min_sc.size:
            constraints += [y >= y_min_sc, y <= y_max_sc]
        if len(y_ibr_idx) and len(ibr_idx):
            reserve_up = pg_max[ibr_idx] - pg[ibr_idx]
            reserve_dn = pg_min[ibr_idx] - pg[ibr_idx]
            reserve_up_sc = cp.multiply(reserve_up, y_scaler.scale_[y_ibr_idx]) + y_scaler.min_[y_ibr_idx]
            reserve_dn_sc = cp.multiply(reserve_dn, y_scaler.scale_[y_ibr_idx]) + y_scaler.min_[y_ibr_idx]
            constraints += [y[y_ibr_idx] <= reserve_up_sc, y[y_ibr_idx] >= reserve_dn_sc]
        block_sizes["output"] = len(constraints) - sum(v for k, v in block_sizes.items() if k != "output")

    if use_nn:
        y_nn, nn_constraints = build_nn_constraints_local(model, x, x_min_sc, x_max_sc)
        constraints += nn_constraints
        if y is None:
            y = cp.Variable(len(y_names), name="y")
        constraints.append(y == y_nn)
        block_sizes["nn"] = len(nn_constraints) + 1

    if use_line or use_n1:
        artifacts = build_line_artifacts_local(ss)
        flows, line_constraints = build_basecase_line_constraints_local(pg, pd, artifacts)
        if use_line:
            constraints += line_constraints
            block_sizes["line"] = len(line_constraints)
        if use_n1:
            n1_constraints, n_active_outages = build_n1_constraints_local(flows, artifacts)
            constraints += n1_constraints
            block_sizes["n1"] = len(n1_constraints)
        else:
            n_active_outages = 0
    else:
        n_active_outages = 0

    if cost_table_path is not None and cost_table_path.exists():
        a, b, c = load_dispatch_cost_arrays_from_table(ss, cost_table_path)
        objective = cp.Minimize(cp.sum(c + cp.multiply(b, pg) + cp.multiply(a, cp.square(pg))))
    else:
        objective = cp.Minimize(cp.sum_squares(pg))

    problem = cp.Problem(objective, constraints)
    has_binaries = bool(use_nn)
    solver_name = CONFIG["solver_mip"] if has_binaries else CONFIG["solver_continuous"]
    try:
        problem.solve(solver=solver_name, verbose=False)
    except Exception as exc:
        return {
            "status": f"solver_error: {type(exc).__name__}",
            "exception": str(exc),
            "missing_features": missing_features,
            "nonfinite_seed": nonfinite_seed,
            "n_constraints": len(constraints),
            "block_sizes": block_sizes,
            "n_active_outages": n_active_outages,
        }

    return {
        "status": problem.status,
        "objective": None if problem.value is None else float(problem.value),
        "missing_features": missing_features,
        "nonfinite_seed": nonfinite_seed,
        "n_constraints": len(constraints),
        "block_sizes": block_sizes,
        "balance_rhs": balance_rhs,
        "sum_pg": None if pg.value is None else float(np.sum(np.asarray(pg.value, dtype=float))),
        "n_active_outages": n_active_outages,
    }


In [ ]:
STAGES = [
    {"name": "A_ed_only", "use_input": False, "use_output": False, "use_nn": False, "use_line": False, "use_n1": False},
    {"name": "B_add_input", "use_input": True, "use_output": False, "use_nn": False, "use_line": False, "use_n1": False},
    {"name": "C_add_output", "use_input": True, "use_output": True, "use_nn": False, "use_line": False, "use_n1": False},
    {"name": "D_add_nn", "use_input": True, "use_output": True, "use_nn": True, "use_line": False, "use_n1": False},
    {"name": "E_add_line", "use_input": True, "use_output": True, "use_nn": True, "use_line": True, "use_n1": False},
    {"name": "F_add_n1", "use_input": True, "use_output": True, "use_nn": True, "use_line": True, "use_n1": True},
]

BALANCE_MODE = "root"  # choose: "tmp" or "root"

rows = []
for stage in STAGES:
    result = solve_debug_stage(balance_mode=BALANCE_MODE, **{k: v for k, v in stage.items() if k != "name"})
    row = {"stage": stage["name"], "bundle": MODEL_BUNDLE, "balance_mode": BALANCE_MODE}
    row.update({k: v for k, v in result.items() if k not in {"block_sizes", "missing_features", "nonfinite_seed", "exception"}})
    row["missing_count"] = len(result.get("missing_features", []))
    row["nonfinite_count"] = len(result.get("nonfinite_seed", []))
    rows.append(row)

ladder_df = pd.DataFrame(rows)
ladder_df


In [ ]:
TARGET_STAGE = "F_add_n1"
stage_cfg = next(item for item in STAGES if item["name"] == TARGET_STAGE)
detail = solve_debug_stage(balance_mode=BALANCE_MODE, **{k: v for k, v in stage_cfg.items() if k != "name"})

print("status:", detail.get("status"))
print("objective:", detail.get("objective"))
print("n_constraints:", detail.get("n_constraints"))
print("balance rhs:", detail.get("balance_rhs"))
print("sum pg:", detail.get("sum_pg"))
print("n active outages:", detail.get("n_active_outages"))
print("block sizes:", detail.get("block_sizes"))
print("missing features sample:", detail.get("missing_features", [])[:20])
print("nonfinite seed sample:", detail.get("nonfinite_seed", [])[:20])
if "exception" in detail:
    print("exception:", detail["exception"])
